In [ ]:
"""
Smart Building Digital Forensic Readiness Framework
Step 1 (IoT-23 version): Train an attack detector on REAL IoT traffic

WHAT THIS SCRIPT DOES (in simple words):
1. Finds the IoT-23 "conn.log.labeled" files (one per capture / "scenario")
2. Reads them in chunks (they are huge) and keeps a random sample of
   benign and malicious flows from each scenario
3. Splits the data BY SCENARIO: the model is tested on captures it has
   never seen. This is the honest way to test on IoT-23 - a random row
   split would put near-identical flows from the same capture in both
   train and test and give fake-high accuracy.
4. Trains a small neural network (MLP) for 10 epochs and prints, for every
   epoch: training loss, training accuracy, validation accuracy,
   test accuracy (plus validation F1 for the malicious class, because
   accuracy alone is misleading when classes are imbalanced)
5. Prints precision / recall / F1 / PR-AUC, a confusion matrix, and a
   breakdown per scenario and per attack type
6. Saves the model, the preprocessing recipe and the split for Step 2

HOW TO RUN (Colab):
   !wget -c https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/iot_23_datasets_small.tar.gz
   !mkdir -p /content/iot23 && tar -xzf iot_23_datasets_small.tar.gz -C /content/iot23
   then run this script. The first run reads all the logs and can take a
   while; the sample is cached so later runs start quickly.
   Tip: set MAX_CHUNKS_PER_FILE = 4 for a quick trial run first.

HOW TO RUN (local):
   Extract the dataset somewhere, set DATA_ROOT, run:
   python iot23_step1_train.py

REQUIREMENTS:
   pip install torch pandas scikit-learn numpy

HONEST NOTES (also worth putting in your report):
 - IoT-23 is real IoT traffic, but the malware was run in a controlled
   network, and the benign data is only a few captures.
 - We cap how many flows we keep per class per scenario, so the
   benign/malicious ratio here is NOT the real-world ratio. In real
   networks attacks are far rarer, so precision here looks better than it
   would in deployment. Compare models using recall, F1 and PR-AUC.
 - Scenarios of the same malware family can land in both train and test.
   For a stricter test, hold out whole families.
"""

import copy
import csv
import glob
import json
import os
import pickle
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (average_precision_score, classification_report,
                             confusion_matrix, f1_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler

# ----------------------------------------------------------------
# SETTINGS - change these if needed
# ----------------------------------------------------------------
DATA_ROOT = "/content/iot23"            # where you extracted the dataset
OUTPUT_DIR_COLAB = "/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs"
OUTPUT_DIR_LOCAL = "./iot23_outputs"

MAX_ROWS_PER_CLASS_PER_SCENARIO = 40_000   # sample size per class, per capture
MAX_CHUNKS_PER_FILE = None                 # None = read whole files; e.g. 4 = quick trial
CHUNK_ROWS = 500_000
MAX_SCENARIOS = None                       # None = use all captures found
DROP_DUPLICATES = True                     # see the note below
# NOTE on DROP_DUPLICATES: IoT-23 attack traffic (port scans etc.) is extremely repetitive -
#   thousands of flows have identical features. True  = score distinct flow patterns only (a
#   harder, more honest test: no credit for repetition, but attack counts shrink a lot).
#   False = count every flow (volume-weighted: scan-heavy captures will push accuracy up).
#   Try both and report which one you used.
USE_CACHE = True                           # reuse the sampled data on later runs
MIN_ROWS_PER_CLASS_PER_SPLIT = 500         # each split must contain both classes

EPOCHS = 10
BATCH_SIZE = 1024
LEARNING_RATE = 1e-3
HIDDEN_DIMS = [128, 64]
RANDOM_STATE = 42

# ----------------------------------------------------------------
# Where do outputs go? (Colab: Google Drive, so they survive restarts)
# ----------------------------------------------------------------
try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    OUTPUT_DIR = OUTPUT_DIR_COLAB
else:
    OUTPUT_DIR = OUTPUT_DIR_LOCAL
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Outputs will be saved to: {OUTPUT_DIR}")

rng = np.random.default_rng(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# ----------------------------------------------------------------
# Which columns of the Zeek conn log we use
#   - IPs, timestamps and uid are NOT used as features (each capture has
#     one infected device, so IPs would let the model just memorise the
#     capture instead of learning attack behaviour)
#   - "detailed-label" (the attack type) is NOT used as a feature either:
#     that would leak the answer
# ----------------------------------------------------------------
NUMERIC_RAW = ["id.resp_p", "duration", "orig_bytes", "resp_bytes", "missed_bytes",
               "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes"]
LOG1P_COLS = ["duration", "orig_bytes", "resp_bytes", "missed_bytes",
              "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes"]
CAT_COLS = ["proto", "service", "conn_state"]
KEEP_COLS = NUMERIC_RAW + CAT_COLS + ["history", "label", "detailed-label"]

# Standard IoT-23 layout, used only if a file has no "#fields" header line
FALLBACK_TAB_NAMES = [
    "ts", "uid", "id.orig_h", "id.orig_p", "id.resp_h", "id.resp_p", "proto",
    "service", "duration", "orig_bytes", "resp_bytes", "conn_state",
    "local_orig", "local_resp", "missed_bytes", "history", "orig_pkts",
    "orig_ip_bytes", "resp_pkts", "resp_ip_bytes",
    "tunnel_parents   label   detailed-label",
]


# ================================================================
# PART 1: Reading the Zeek logs
# ================================================================
def find_conn_files(root):
    paths = glob.glob(os.path.join(root, "**", "*conn*labeled"), recursive=True)
    return sorted(p for p in paths if os.path.isfile(p))


def scenario_name(path):
    parent = os.path.basename(os.path.dirname(path))
    if parent.lower() in ("bro", "zeek"):
        parent = os.path.basename(os.path.dirname(os.path.dirname(path)))
    return parent


def _read_header(path):
    fields_line, first_data = None, None
    with open(path, "r", errors="replace") as f:
        for line in f:
            if line.startswith("#fields"):
                fields_line = line.rstrip("\n")
            elif not line.startswith("#"):
                first_data = line.rstrip("\n")
                break
    return fields_line, first_data


def _merge_topk(kept_df, kept_keys, new_df, new_keys, k):
    """Keep a uniform random sample of size k from a stream of chunks:
    give every row a random number and keep the k rows with the smallest."""
    if kept_df is None:
        cat, keys = new_df, new_keys
    else:
        cat = pd.concat([kept_df, new_df], ignore_index=True)
        keys = np.concatenate([kept_keys, new_keys])
    if len(cat) > k:
        idx = np.argpartition(keys, k)[:k]
        cat, keys = cat.iloc[idx].reset_index(drop=True), keys[idx]
    return cat, keys


def read_zeek_log(path, required_cols, optional_cols=(), per_class_cap=None,
                  max_chunks=None, max_rows=None):
    """Read one Zeek conn log. Handles both layouts seen in IoT-23:
       (a) last three fields joined by spaces: 'tunnel_parents   label   detailed-label'
       (b) every field tab-separated.
    If per_class_cap is given, returns a random sample of at most that many
    benign AND that many malicious flows. Returns (DataFrame, stats)."""
    fields_line, first_data = _read_header(path)
    stats = {"seen_benign": 0, "seen_malicious": 0, "bad_label": 0, "rows_read": 0}
    if first_data is None:
        return pd.DataFrame(columns=list(required_cols)), stats

    tab_names = (fields_line.split("\t")[1:] if fields_line is not None
                 else list(FALLBACK_TAB_NAMES))
    flat_names = [t for name in tab_names for t in name.split()]
    ncols = len(first_data.split("\t"))

    if ncols == len(flat_names):
        col_names, tail_names = flat_names, None
    elif ncols == len(tab_names):
        col_names, tail_names = tab_names[:-1] + ["__tail__"], tab_names[-1].split()
    else:
        raise ValueError(
            f"Unexpected layout in {path}\n  header has {len(tab_names)} tab fields "
            f"({len(flat_names)} names), but the first data row has {ncols} fields.\n"
            f"  first data row: {first_data[:200]}")

    reader = pd.read_csv(
        path, sep="\t", header=None, names=col_names, comment="#", dtype=str,
        na_filter=False, quoting=csv.QUOTE_NONE, on_bad_lines="skip",
        chunksize=CHUNK_ROWS, engine="c")

    kept = {0: None, 1: None}
    kept_keys = {0: None, 1: None}
    collected = []
    for ci, chunk in enumerate(reader):
        if max_chunks is not None and ci >= max_chunks:
            break
        chunk = chunk.reset_index(drop=True)
        if tail_names is not None:
            tail = (chunk.pop("__tail__").str.split(n=len(tail_names) - 1, expand=True)
                    .reindex(columns=range(len(tail_names))))
            for i, name in enumerate(tail_names):
                chunk[name] = tail[i].fillna("-").values
        missing = [c for c in required_cols if c not in chunk.columns]
        if missing:
            raise ValueError(f"{path} is missing expected columns: {missing}")
        cols = list(required_cols) + [c for c in optional_cols if c in chunk.columns]
        sub = chunk[cols]
        stats["rows_read"] += len(sub)

        if per_class_cap is not None:
            lab = sub["label"].str.strip().str.lower()
            valid = lab.isin(["benign", "malicious"]).values
            stats["bad_label"] += int((~valid).sum())
            sub, lab = sub[valid], lab[valid]
            is_ben = lab.eq("benign").values
            for cls, mask in ((0, is_ben), (1, ~is_ben)):
                part = sub[mask]
                stats["seen_benign" if cls == 0 else "seen_malicious"] += len(part)
                if len(part) == 0:
                    continue
                keys = rng.random(len(part))
                kept[cls], kept_keys[cls] = _merge_topk(
                    kept[cls], kept_keys[cls], part, keys, per_class_cap)
        else:
            collected.append(sub)
            if max_rows is not None and stats["rows_read"] >= max_rows:
                break

    if per_class_cap is not None:
        parts = [k for k in kept.values() if k is not None]
        df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=list(required_cols))
    else:
        df = pd.concat(collected, ignore_index=True) if collected else pd.DataFrame(columns=list(required_cols))
        if max_rows is not None:
            df = df.head(max_rows)
    return df, stats


# ================================================================
# PART 2: Turning flows into model inputs
# (Step 2 uses the exact same recipe - keep both files identical here)
# ================================================================
def numeric_frame(df):
    out = pd.DataFrame(index=df.index)
    for c in NUMERIC_RAW:
        out[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).clip(lower=0.0)
    # "history" is Zeek's string of connection events; '-' means none
    out["history_len"] = df["history"].astype(str).str.replace("-", "", regex=False).str.len()
    for c in LOG1P_COLS:
        out[c] = np.log1p(out[c])
    return out


def make_matrix(num_part, cat_part, scaler, cat_vocab):
    parts = [scaler.transform(num_part.values)]
    names = list(num_part.columns)
    for c in CAT_COLS:
        s = cat_part[c].astype(str).values
        for v in cat_vocab[c]:
            parts.append((s == v).astype(np.float32)[:, None])
            names.append(f"{c}={v}")
    return np.hstack(parts).astype(np.float32), names


# ================================================================
# PART 3: Load (or build) the sample
# ================================================================
cache_path = os.path.join(
    OUTPUT_DIR, f"iot23_sample_cap{MAX_ROWS_PER_CLASS_PER_SCENARIO}_chunks{MAX_CHUNKS_PER_FILE}_sc{MAX_SCENARIOS}.pkl")

if USE_CACHE and os.path.exists(cache_path):
    print(f"\nLoading cached sample from {cache_path}")
    with open(cache_path, "rb") as f:
        cached = pickle.load(f)
    big, scenario_stats = cached["df"], cached["stats"]
else:
    files = find_conn_files(DATA_ROOT)
    if not files:
        print(f"\n\u26a0\ufe0f  No 'conn.log.labeled' files found under: {DATA_ROOT}")
        if os.path.isdir(DATA_ROOT):
            print("Top-level contents of that folder:")
            for item in os.listdir(DATA_ROOT)[:30]:
                print("  -", item)
            print("\nIf the dataset is extracted elsewhere, set DATA_ROOT to that folder.")
        else:
            print("That folder doesn't exist. Download + extract the dataset first (see the top of this file).")
        raise SystemExit("Fix DATA_ROOT, then re-run.")

    if MAX_SCENARIOS is not None:
        files = [str(p) for p in rng.permutation(files)[:MAX_SCENARIOS]]
    print(f"\nFound {len(files)} capture file(s). Reading and sampling...")

    frames, scenario_stats, used_names = [], {}, set()
    for n, path in enumerate(files, start=1):
        name = scenario_name(path)
        if name in used_names:
            name = f"{name}#{n}"
        used_names.add(name)
        t0 = time.time()
        df_s, st = read_zeek_log(
            path, required_cols=KEEP_COLS, per_class_cap=MAX_ROWS_PER_CLASS_PER_SCENARIO,
            max_chunks=MAX_CHUNKS_PER_FILE)
        if len(df_s) == 0:
            print(f"  [{n}/{len(files)}] {name}: no usable rows, skipped")
            continue
        df_s["scenario"] = name
        frames.append(df_s)
        scenario_stats[name] = st
        print(f"  [{n}/{len(files)}] {name}: read {st['rows_read']:,} flows "
              f"(benign {st['seen_benign']:,} / malicious {st['seen_malicious']:,}), "
              f"kept {len(df_s):,}  ({time.time() - t0:.0f}s)")
        if st["bad_label"]:
            print(f"      note: {st['bad_label']:,} rows had an unexpected label and were dropped")

    if not frames:
        raise SystemExit("No usable data was read. Check the file format and DATA_ROOT.")
    big = pd.concat(frames, ignore_index=True)
    with open(cache_path, "wb") as f:
        pickle.dump({"df": big, "stats": scenario_stats}, f)
    print(f"Saved sample cache to {cache_path}")

# Labels: Benign -> 0, anything else (Malicious) -> 1
big["y"] = (big["label"].str.strip().str.lower() != "benign").astype(int)
big["detail"] = np.where(big["y"] == 0, "(benign)", big["detailed-label"])
big.loc[(big["y"] == 1) & (big["detail"] == "-"), "detail"] = "(unspecified)"

num_df = numeric_frame(big)

if DROP_DUPLICATES:
    before = len(big)
    key = pd.concat([num_df, big[CAT_COLS], big[["scenario", "y"]]], axis=1)
    keep = ~key.duplicated().values
    big, num_df = big[keep].reset_index(drop=True), num_df[keep].reset_index(drop=True)
    print(f"\nRemoved {before - len(big):,} duplicate flows ({before:,} -> {len(big):,}).")

counts = (big.groupby(["scenario", "y"]).size().unstack(fill_value=0)
          .reindex(columns=[0, 1], fill_value=0))
counts.columns = ["benign", "malicious"]
print(f"\nSample overall: {len(big):,} flows | benign {int(counts['benign'].sum()):,} | "
      f"malicious {int(counts['malicious'].sum()):,}")


# ================================================================
# PART 4: Split BY SCENARIO (test = captures never seen in training)
# ================================================================
def make_scenario_split(counts, seed, min_rows, max_tries=3000):
    scenarios = list(counts.index)
    total = counts.values.sum()
    r = np.random.default_rng(seed)
    for _ in range(max_tries):
        order = r.permutation(scenarios)
        assign, cum = {}, 0
        for s in order:
            if cum < 0.70 * total:
                assign[s] = "train"
            elif cum < 0.85 * total:
                assign[s] = "val"
            else:
                assign[s] = "test"
            cum += counts.loc[s].sum()
        ok = True
        for split in ("train", "val", "test"):
            members = [s for s, v in assign.items() if v == split]
            if (not members or counts.loc[members, "benign"].sum() < min_rows
                    or counts.loc[members, "malicious"].sum() < min_rows):
                ok = False
                break
        if ok:
            return assign
    return None


assign = make_scenario_split(counts, RANDOM_STATE, MIN_ROWS_PER_CLASS_PER_SPLIT)
if assign is None:
    raise SystemExit(
        "Could not find a scenario split where train, validation AND test each contain enough benign "
        "and malicious flows.\nTry: MAX_SCENARIOS=None (use all captures), lower MIN_ROWS_PER_CLASS_PER_SPLIT, "
        "raise MAX_ROWS_PER_CLASS_PER_SCENARIO, or set DROP_DUPLICATES=False (duplicate removal "
        "can shrink the attack class a lot).")

split_of = big["scenario"].map(assign)
train_mask, val_mask, test_mask = (split_of == "train").values, (split_of == "val").values, (split_of == "test").values

print("\nScenario split (the model is tested on captures it never saw):")
for split in ("train", "val", "test"):
    members = [s for s, v in assign.items() if v == split]
    b, m = counts.loc[members, "benign"].sum(), counts.loc[members, "malicious"].sum()
    print(f"  {split:5s}: {len(members):2d} scenarios | {b + m:>9,} flows | benign {b:,} | malicious {m:,}")
    print(f"         {', '.join(sorted(members))}")

# ================================================================
# PART 5: Preprocessing recipe (fit on TRAIN only)
# ================================================================
scaler = StandardScaler().fit(num_df[train_mask].values)
cat_vocab = {c: big.loc[train_mask, c].astype(str).value_counts().head(30).index.tolist() for c in CAT_COLS}

X_all, feature_names = make_matrix(num_df, big, scaler, cat_vocab)
y_all = big["y"].values.astype(np.int64)

X_train, y_train = torch.tensor(X_all[train_mask]), torch.tensor(y_all[train_mask])
X_val, y_val_np = torch.tensor(X_all[val_mask]), y_all[val_mask]
X_test, y_test_np = torch.tensor(X_all[test_mask]), y_all[test_mask]
print(f"\nInput features per flow: {X_all.shape[1]}")


# ================================================================
# PART 6: The model
# ================================================================
class AttackDetectorMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, num_classes):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.2)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


model = AttackDetectorMLP(X_all.shape[1], HIDDEN_DIMS, 2).to(device)
print(f"\nModel architecture:\n{model}")

# Classes are imbalanced, so mistakes on the rarer class count for more
n_train = len(y_train)
n_pos = int((y_train == 1).sum())
n_neg = n_train - n_pos
class_weights = torch.tensor([n_train / (2 * max(n_neg, 1)), n_train / (2 * max(n_pos, 1))],
                             dtype=torch.float32).to(device)
print(f"Class weights (benign, malicious): {class_weights.cpu().numpy().round(3)}")
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


def iterate_batches(X, y, batch_size, shuffle):
    idx = torch.randperm(X.shape[0]) if shuffle else torch.arange(X.shape[0])
    for i in range(0, len(idx), batch_size):
        b = idx[i:i + batch_size]
        yield X[b], y[b]


def predict_malicious_prob(X, batch_size=8192):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            logits = model(X[i:i + batch_size].to(device))
            out.append(torch.softmax(logits, dim=1)[:, 1].cpu())
    return torch.cat(out).numpy()


# ================================================================
# PART 7: Training - 10 epochs, metrics printed every epoch
# ================================================================
print(f"\nTraining for {EPOCHS} epochs...\n")
history, best_val_f1, best_epoch, best_state = [], -1.0, 0, None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in iterate_batches(X_train, y_train, BATCH_SIZE, shuffle=True):
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        correct += (out.argmax(dim=1) == yb).sum().item()
        total += xb.size(0)

    train_loss, train_acc = running_loss / total, correct / total
    val_pred = (predict_malicious_prob(X_val) >= 0.5).astype(int)
    test_pred_e = (predict_malicious_prob(X_test) >= 0.5).astype(int)
    val_acc = float((val_pred == y_val_np).mean())
    test_acc = float((test_pred_e == y_test_np).mean())
    val_f1 = float(f1_score(y_val_np, val_pred, pos_label=1, zero_division=0))

    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                    "val_acc": val_acc, "test_acc": test_acc, "val_f1_malicious": val_f1})
    if val_f1 > best_val_f1:
        best_val_f1, best_epoch, best_state = val_f1, epoch, copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch:2d}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Train Acc: {train_acc * 100:6.2f}% | Val Acc: {val_acc * 100:6.2f}% | "
          f"Test Acc: {test_acc * 100:6.2f}% | Val F1 (malicious): {val_f1:.4f}")

print("\n(Test accuracy is shown each epoch only because it was asked for. The model that is saved")
print(f" and reported below is the one from the epoch with the best VALIDATION F1: epoch {best_epoch}.)")
model.load_state_dict(best_state)

# ================================================================
# PART 8: Final, honest evaluation on the unseen scenarios
# ================================================================
p_test = predict_malicious_prob(X_test)
test_pred = (p_test >= 0.5).astype(int)

print("\n" + "=" * 70)
print("FINAL TEST RESULTS (scenarios never seen during training)")
print("=" * 70)
report_text = classification_report(y_test_np, test_pred, labels=[0, 1],
                                    target_names=["benign", "malicious"], digits=4, zero_division=0)
print(report_text)
report_dict = classification_report(y_test_np, test_pred, labels=[0, 1],
                                    target_names=["benign", "malicious"], output_dict=True, zero_division=0)
cm = confusion_matrix(y_test_np, test_pred, labels=[0, 1])
print("Confusion matrix (rows = actual, columns = predicted):")
print(f"                 pred benign   pred malicious")
print(f"  actual benign   {cm[0, 0]:>11,}   {cm[0, 1]:>14,}")
print(f"  actual malicious{cm[1, 0]:>11,}   {cm[1, 1]:>14,}")

pr_auc = float(average_precision_score(y_test_np, p_test))
try:
    roc_auc = float(roc_auc_score(y_test_np, p_test))
except ValueError:
    roc_auc = None
print(f"\nPR-AUC (malicious class): {pr_auc:.4f}" + (f"   ROC-AUC: {roc_auc:.4f}" if roc_auc is not None else ""))

meta = big.loc[test_mask, ["scenario", "detail", "y"]].reset_index(drop=True)
meta["pred"] = test_pred

print("\nPer-scenario results on the test set (recall = attacks caught, FP rate = benign flagged):")
rows = []
for sc, g in meta.groupby("scenario"):
    ben, mal = g[g.y == 0], g[g.y == 1]
    rows.append({"scenario": sc, "flows": len(g), "benign": len(ben), "malicious": len(mal),
                 "accuracy": (g.pred == g.y).mean(),
                 "recall": mal.pred.mean() if len(mal) else np.nan,
                 "fp_rate": ben.pred.mean() if len(ben) else np.nan})
print(pd.DataFrame(rows).round(3).to_string(index=False))

print("\nPer attack type on the test set (share of flows the model flagged as attack):")
by_detail = (meta.groupby("detail").agg(flows=("y", "size"), flagged_as_attack=("pred", "mean"))
             .sort_values("flows", ascending=False))
by_detail["flagged_as_attack"] = (by_detail["flagged_as_attack"] * 100).round(1).astype(str) + "%"
print(by_detail.to_string())
print("  (for '(benign)' a LOW percentage is good; for attack types a HIGH percentage is good)")

# ================================================================
# PART 9: Save everything Step 2 needs
# ================================================================
torch.save({"model_state_dict": model.state_dict(), "input_dim": X_all.shape[1],
            "hidden_dims": HIDDEN_DIMS, "num_classes": 2, "feature_names": feature_names,
            "best_epoch": best_epoch}, os.path.join(OUTPUT_DIR, "iot23_model.pt"))

with open(os.path.join(OUTPUT_DIR, "iot23_preprocess.pkl"), "wb") as f:
    pickle.dump({"numeric_cols": list(num_df.columns), "cat_vocab": cat_vocab,
                 "scaler": scaler, "feature_names": feature_names}, f)

with open(os.path.join(OUTPUT_DIR, "iot23_split.json"), "w") as f:
    json.dump(assign, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "iot23_history.pkl"), "wb") as f:
    pickle.dump(history, f)

with open(os.path.join(OUTPUT_DIR, "iot23_metrics.json"), "w") as f:
    json.dump({"best_epoch": best_epoch, "classification_report": report_dict,
               "confusion_matrix": cm.tolist(), "pr_auc_malicious": pr_auc, "roc_auc": roc_auc,
               "test_flows": int(len(y_test_np)), "per_epoch": history}, f, indent=2)

print(f"\n\u2705 Saved to {OUTPUT_DIR}:")
for fn in ("iot23_model.pt", "iot23_preprocess.pkl", "iot23_split.json", "iot23_history.pkl", "iot23_metrics.json"):
    print("   -", fn)
print("\nDone! Next: run iot23_step2_detect_and_log.py on a capture the model has NOT seen (see iot23_split.json).")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu
Outputs will be saved to: /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs

⚠️  No 'conn.log.labeled' files found under: /content/iot23
That folder doesn't exist. Download + extract the dataset first (see the top of this file).


SystemExit: Fix DATA_ROOT, then re-run.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
!df -h /content
!wget -c https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/iot_23_datasets_small.tar.gz -P /content
!mkdir -p /content/iot23
!tar -xzf /content/iot_23_datasets_small.tar.gz -C /content/iot23

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  20% /
--2026-10-11 15:44:46--  https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/iot_23_datasets_small.tar.gz
Resolving mcfp.felk.cvut.cz (mcfp.felk.cvut.cz)... 147.32.82.194
Connecting to mcfp.felk.cvut.cz (mcfp.felk.cvut.cz)|147.32.82.194|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9373916249 (8.7G) [application/x-gzip]
Saving to: ‘/content/iot_23_datasets_small.tar.gz’

iot_23_datasets_sma 100%[===================>]   8.73G  16.3MB/s    in 10m 31s 

2026-10-11 15:55:18 (14.2 MB/s) - ‘/content/iot_23_datasets_small.tar.gz’ saved [9373916249/9373916249]

!find /content/iot23 -name "*conn*labeled" | sort


In [ ]:
!find /content/iot23 -name "*conn*labeled" | sort
!du -sh /content/iot23

/content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-Honeypot-Capture-4-1/bro/conn.log.labeled
/content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-Honeypot-Capture-5-1/bro/conn.log.labeled
/content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-Honeypot-Capture-7-1/Somfy-01/bro/conn.log.labeled
/content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-1-1/bro/conn.log.labeled
/content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-17-1/bro/conn.log.labeled
/content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-20-1/bro/conn.log.labeled
/content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-21-1/bro/conn.log.labeled
/content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-3-1/bro/conn.log.labeled
/content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-33-1/bro/conn.log.labeled
/content/iot23/opt/Malwa

In [ ]:
!sed -i 's/^MAX_CHUNKS_PER_FILE = None/MAX_CHUNKS_PER_FILE = 4/' iot23_step1_train.py
%run iot23_step1_train.py

sed: can't read iot23_step1_train.py: No such file or directory


Exception: File `'iot23_step1_train.py'` not found.

In [ ]:
import glob
for p in glob.glob("/content/**/iot23_step1_train.py", recursive=True) + \
         glob.glob("/content/drive/MyDrive/**/iot23_step1_train.py", recursive=True):
    print(p)

In [ ]:
from google.colab import files; files.upload()

Saving iot23_step1_train.py to iot23_step1_train.py


{'iot23_step1_train.py': b'"""\nSmart Building Digital Forensic Readiness Framework\nStep 1 (IoT-23 version): Train an attack detector on REAL IoT traffic\n\nWHAT THIS SCRIPT DOES (in simple words):\n1. Finds the IoT-23 "conn.log.labeled" files (one per capture / "scenario")\n2. Reads them in chunks (they are huge) and keeps a random sample of\n   benign and malicious flows from each scenario\n3. Splits the data BY SCENARIO: the model is tested on captures it has\n   never seen. This is the honest way to test on IoT-23 - a random row\n   split would put near-identical flows from the same capture in both\n   train and test and give fake-high accuracy.\n4. Trains a small neural network (MLP) for 10 epochs and prints, for every\n   epoch: training loss, training accuracy, validation accuracy,\n   test accuracy (plus validation F1 for the malicious class, because\n   accuracy alone is misleading when classes are imbalanced)\n5. Prints precision / recall / F1 / PR-AUC, a confusion matrix, a

In [ ]:
!wget -c https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/iot_23_datasets_small.tar.gz
!mkdir -p /content/iot23 && tar -xzf iot_23_datasets_small.tar.gz -C /content/iot23

--2026-10-11 16:20:53--  https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/iot_23_datasets_small.tar.gz
Resolving mcfp.felk.cvut.cz (mcfp.felk.cvut.cz)... 147.32.82.194
Connecting to mcfp.felk.cvut.cz (mcfp.felk.cvut.cz)|147.32.82.194|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.



In [ ]:
import glob
files = glob.glob("/content/iot23/**/*conn*labeled", recursive=True)
print(len(files), "files found")
print(files[:3])

23 files found
['/content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-34-1/bro/conn.log.labeled', '/content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-60-1/bro/conn.log.labeled', '/content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-8-1/bro/conn.log.labeled']


In [ ]:
!sed -i 's/^MAX_CHUNKS_PER_FILE = None/MAX_CHUNKS_PER_FILE = 4/' iot23_step1_train.py
!grep -n "^MAX_CHUNKS_PER_FILE" iot23_step1_train.py
%run iot23_step1_train.py

70:MAX_CHUNKS_PER_FILE = 4                 # None = read whole files; e.g. 4 = quick trial
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu
Outputs will be saved to: /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs

Found 23 capture file(s). Reading and sampling...
  [1/23] CTU-Honeypot-Capture-4-1: read 452 flows (benign 452 / malicious 0), kept 452  (0s)
  [2/23] CTU-Honeypot-Capture-5-1: read 1,374 flows (benign 1,374 / malicious 0), kept 1,374  (0s)
  [3/23] Somfy-01: read 130 flows (benign 130 / malicious 0), kept 130  (0s)
  [4/23] CTU-IoT-Malware-Capture-1-1: read 1,008,748 flows (benign 469,275 / malicious 539,473), kept 80,000  (17s)
  [5/23] CTU-IoT-Malware-Capture-17-1: read 2,000,000 flows (benign 1,037 / malicious 1,998,963), kept 41,037  (21s)
  [6/23] CTU-IoT-Malware-Capture-20-1: read 3,209 flows (benign 3,193 / malicious 16), kept 3,209  (0s)
  [7/

In [ ]:
!sed -i 's/^MAX_CHUNKS_PER_FILE = 4/MAX_CHUNKS_PER_FILE = None/' iot23_step1_train.py
!grep -n "^MAX_CHUNKS_PER_FILE" iot23_step1_train.py
%run iot23_step1_train.py

70:MAX_CHUNKS_PER_FILE = None                 # None = read whole files; e.g. 4 = quick trial
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu
Outputs will be saved to: /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs

Found 23 capture file(s). Reading and sampling...
  [1/23] CTU-Honeypot-Capture-4-1: read 452 flows (benign 452 / malicious 0), kept 452  (0s)
  [2/23] CTU-Honeypot-Capture-5-1: read 1,374 flows (benign 1,374 / malicious 0), kept 1,374  (0s)
  [3/23] Somfy-01: read 130 flows (benign 130 / malicious 0), kept 130  (0s)
  [4/23] CTU-IoT-Malware-Capture-1-1: read 1,008,748 flows (benign 469,275 / malicious 539,473), kept 80,000  (16s)
  [5/23] CTU-IoT-Malware-Capture-17-1: read 54,659,855 flows (benign 31,438 / malicious 54,628,417), kept 71,438  (623s)
  [6/23] CTU-IoT-Malware-Capture-20-1: read 3,209 flows (benign 3,193 / malicious 16), kept 3,209  (0s

In [ ]:
import os
out = "/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs"
print(os.path.exists(f"{out}/iot23_sample_cap40000_chunksNone_scNone.pkl"))

True


In [ ]:
import re, shutil
SAVE_DIR = "/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs"
src = open("iot23_step1_train.py").read()
for seed in [1, 2, 3, 4, 5]:
    open("run_tmp.py", "w").write(re.sub(r"^RANDOM_STATE = \d+", f"RANDOM_STATE = {seed}", src, flags=re.M))
    get_ipython().run_line_magic("run", "run_tmp.py")
    for fn in ("iot23_metrics.json", "iot23_split.json"):
        shutil.copy(f"{SAVE_DIR}/{fn}", f"{SAVE_DIR}/seed{seed}_{fn}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu
Outputs will be saved to: /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs

Loading cached sample from /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/iot23_sample_cap40000_chunksNone_scNone.pkl

Removed 733,673 duplicate flows (886,853 -> 153,180).

Sample overall: 153,180 flows | benign 84,981 | malicious 68,199

Scenario split (the model is tested on captures it never saw):
  train: 12 scenarios |   118,278 flows | benign 56,046 | malicious 62,232
         CTU-Honeypot-Capture-4-1, CTU-IoT-Malware-Capture-1-1, CTU-IoT-Malware-Capture-17-1, CTU-IoT-Malware-Capture-3-1, CTU-IoT-Malware-Capture-33-1, CTU-IoT-Malware-Capture-35-1, CTU-IoT-Malware-Capture-36-1, CTU-IoT-Malware-Capture-48-1, CTU-IoT-Malware-Capture-49-1, CTU-IoT-Malware-Capture-52-1, CTU-IoT-Malware-Capture-60-1, Somfy-01
  val

In [ ]:
import json
SAVE_DIR = "/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs"
for s in [1, 2, 3, 4, 5]:
    m = json.load(open(f"{SAVE_DIR}/seed{s}_iot23_metrics.json"))
    r = m["classification_report"]["malicious"]
    split = json.load(open(f"{SAVE_DIR}/seed{s}_iot23_split.json"))
    test = [k for k, v in split.items() if v == "test"]
    print(f"seed {s}: recall {r['recall']:.3f}  precision {r['precision']:.3f}  "
          f"F1 {r['f1-score']:.3f}  PR-AUC {m['pr_auc_malicious']:.3f}  test={len(test)} scenarios")

seed 1: recall 0.271  precision 0.505  F1 0.353  PR-AUC 0.665  test=8 scenarios
seed 2: recall 0.994  precision 0.986  F1 0.990  PR-AUC 0.944  test=2 scenarios
seed 3: recall 0.833  precision 0.798  F1 0.815  PR-AUC 0.786  test=7 scenarios
seed 4: recall 0.018  precision 0.211  F1 0.034  PR-AUC 0.721  test=7 scenarios
seed 5: recall 0.000  precision 0.000  F1 0.000  PR-AUC 0.353  test=5 scenarios


In [ ]:
from google.colab import files
files.upload()   # choose iot23_step2_detect_and_log.py

Saving iot23_step2_detect_and_log.py to iot23_step2_detect_and_log.py


{'iot23_step2_detect_and_log.py': b'"""\nSmart Building Digital Forensic Readiness Framework\nStep 2 (IoT-23 version): Detect attacks in a Zeek log and keep a tamper-evident log\n\nWHAT THIS SCRIPT DOES (in simple words):\n1. Loads the model + preprocessing recipe saved by iot23_step1_train.py\n2. Reads a Zeek connection log (a labeled IoT-23 file, OR a normal live\n   Zeek conn.log with no labels - the script handles both)\n3. Scores every flow: probability that it is malicious\n4. Every flow above ALERT_THRESHOLD becomes a "detection" entry holding\n   the evidence an investigator needs: flow time, source/destination\n   IP and port, protocol, service, confidence\n5. Each entry is hashed together with the previous entry\'s hash (a hash\n   chain) and appended to detection_log.jsonl. If anyone edits or\n   deletes an old entry later, the chain breaks and verification fails.\n   (Same idea as the chain-of-custody design in the project plan.)\n6. Verifies the whole chain at the end, and

In [ ]:
import json
SAVE_DIR = "/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs"
split = json.load(open(f"{SAVE_DIR}/iot23_split.json"))
print([k for k, v in split.items() if v == "test"])

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/iot23_split.json'

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
SAVE_DIR = "/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs"
print(os.path.exists(SAVE_DIR))
print(sorted(os.listdir(SAVE_DIR)) if os.path.exists(SAVE_DIR) else "folder not found")

Mounted at /content/drive
True
['iot23_history.pkl', 'iot23_metrics.json', 'iot23_model.pt', 'iot23_preprocess.pkl', 'iot23_sample_cap40000_chunks4_scNone.pkl', 'iot23_sample_cap40000_chunksNone_scNone.pkl', 'iot23_split.json', 'seed1_iot23_metrics.json', 'seed1_iot23_split.json', 'seed2_iot23_metrics.json', 'seed2_iot23_split.json', 'seed3_iot23_metrics.json', 'seed3_iot23_split.json', 'seed4_iot23_metrics.json', 'seed4_iot23_split.json', 'seed5_iot23_metrics.json', 'seed5_iot23_split.json']


In [ ]:
import json
SAVE_DIR = "/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs"
split = json.load(open(f"{SAVE_DIR}/iot23_split.json"))
print([k for k, v in split.items() if v == "test"])

['CTU-IoT-Malware-Capture-8-1', 'CTU-IoT-Malware-Capture-9-1', 'CTU-Honeypot-Capture-5-1', 'CTU-IoT-Malware-Capture-7-1', 'CTU-IoT-Malware-Capture-52-1']


In [ ]:
import os, glob
print("step2 script:", os.path.exists("iot23_step2_detect_and_log.py"))
print("dataset files:", len(glob.glob("/content/iot23/**/*conn*labeled", recursive=True)))

step2 script: True
dataset files: 0


In [ ]:
import json
SAVE_DIR = "/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs"
split = json.load(open(f"{SAVE_DIR}/iot23_split.json"))
print([k for k, v in split.items() if v == "test"])

['CTU-IoT-Malware-Capture-8-1', 'CTU-IoT-Malware-Capture-9-1', 'CTU-Honeypot-Capture-5-1', 'CTU-IoT-Malware-Capture-7-1', 'CTU-IoT-Malware-Capture-52-1']


In [ ]:
CAPTURE = "CTU-IoT-Malware-Capture-34-1"   # change to your pick
!wget -c https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/iot_23_datasets_small.tar.gz
!mkdir -p /content/iot23 && tar -xzf iot_23_datasets_small.tar.gz -C /content/iot23 --wildcards "*{CAPTURE}/bro/conn.log.labeled"

--2026-10-12 04:52:06--  https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/iot_23_datasets_small.tar.gz
Resolving mcfp.felk.cvut.cz (mcfp.felk.cvut.cz)... 147.32.82.194
Connecting to mcfp.felk.cvut.cz (mcfp.felk.cvut.cz)|147.32.82.194|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9373916249 (8.7G) [application/x-gzip]
Saving to: ‘iot_23_datasets_small.tar.gz’

iot_23_datasets_sma 100%[===================>]   8.73G  8.50MB/s    in 26m 57s 

2026-10-12 05:19:04 (5.53 MB/s) - ‘iot_23_datasets_small.tar.gz’ saved [9373916249/9373916249]



In [ ]:
import glob
print(glob.glob(f"/content/iot23/**/{CAPTURE}/**/conn.log.labeled", recursive=True))

['/content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-34-1/bro/conn.log.labeled']


In [ ]:
import re, glob
CAPTURE = "CTU-IoT-Malware-Capture-34-1"
matches = glob.glob(f"/content/iot23/**/{CAPTURE}/**/conn.log.labeled", recursive=True)
assert len(matches) == 1, "capture not found, check the name"

code = open("iot23_step2_detect_and_log.py").read()
code = re.sub(r"^INPUT_LOG = .*$", f'INPUT_LOG = "{matches[0]}"', code, flags=re.M)
code = re.sub(r"^ALERT_THRESHOLD = [0-9.]+", "ALERT_THRESHOLD = 0.2", code, flags=re.M)  # delete this line to keep 0.5
open("iot23_step2_detect_and_log.py", "w").write(code)

%run iot23_step2_detect_and_log.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu
Loading trained model and preprocessing recipe...

⚠️  'CTU-IoT-Malware-Capture-34-1' was part of the TRAINING data. Results on it are NOT a fair test.

Reading Zeek log: /content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-34-1/bro/conn.log.labeled
Loaded 23,145 flows.
Scoring flows...

Appending detections to: /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/detection_log.jsonl
(flows with malicious probability >= 0.2 are logged; normal traffic is not)

[2018-12-21T14:50:16+00:00] ATTACK suspected: 192.168.1.195 -> 185.244.25.235:80 (tcp/-, S0) confidence 28.1%  -> 365af84b...
[2018-12-21T14:50:48+00:00] ATTACK suspected: 192.168.1.195 -> 185.244.25.235:80 (tcp/http, SF) confidence 91.0%  -> 7dd98bcb...
[2018-12-21T14:50:50+00:00] ATTACK suspected: 192.168.1.195 -> 185.244.25.235:80 (t

In [ ]:
from google.colab import files
files.upload()    # choose iot23_step1b_train_loso.py

Saving iot23_step1b_train_loso.py to iot23_step1b_train_loso.py


{'iot23_step1b_train_loso.py': b'"""\nSmart Building Digital Forensic Readiness Framework\nStep 1b (IoT-23): Train the attack detector with LEAVE-ONE-CAPTURE-OUT testing\n\nWHY THIS SCRIPT EXISTS (in simple words):\n  Step 1 used ONE random split of the 23 captures into train/val/test. The\n  results swung from 0% to 99% attacks caught depending on which captures\n  happened to land in "test". One split is a lottery.\n\n  Here, every capture takes a turn as the test set while the model trains on\n  all the others. The predictions from all turns are pooled, so you get one\n  stable score plus a per-capture breakdown that shows exactly where the model\n  fails.\n\nWHAT IT DOES:\n  1. Loads the sampled flows that Step 1 cached (no need to re-read the huge logs)\n  2. Keeps a few captures completely aside as the FINAL, untouched test set\n     (DEMO_HOLDOUT). Nothing is tuned on them.\n  3. Leave-one-capture-out on the rest, for a few model variants:\n       baseline    : same features as 

In [ ]:
%run iot23_step1b_train_loso.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu
Outputs will be saved to: /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs

Loading cached sample from /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/iot23_sample_cap40000_chunksNone_scNone.pkl
Removed 733,673 duplicate flows (886,853 -> 153,180).
Sample: 153,180 flows | benign 84,981 | malicious 68,199 | 23 captures

Final holdout (never used for any choice): ['CTU-IoT-Malware-Capture-34-1', 'CTU-IoT-Malware-Capture-1-1']
Leave-one-out folds (8): CTU-IoT-Malware-Capture-17-1, CTU-IoT-Malware-Capture-3-1, CTU-IoT-Malware-Capture-35-1, CTU-IoT-Malware-Capture-48-1, CTU-IoT-Malware-Capture-49-1, CTU-IoT-Malware-Capture-52-1, CTU-IoT-Malware-Capture-7-1, CTU-IoT-Malware-Capture-8-1

VARIANT 'baseline'  (drop nothing, weight decay 0.0)
  fold  1/8 held out CTU-IoT-Malware-Capture-17-1: 4,836 fl

In [ ]:
import os, re, glob, shutil
SAVE_DIR = "/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs"

# keep the old training-capture demo log separate
old = f"{SAVE_DIR}/detection_log.jsonl"
if os.path.exists(old):
    shutil.move(old, f"{SAVE_DIR}/detection_log_trainingcapture_demo.jsonl")

CAPTURE = "CTU-IoT-Malware-Capture-34-1"
matches = glob.glob(f"/content/iot23/**/{CAPTURE}/**/conn.log.labeled", recursive=True)
assert len(matches) == 1, "capture not found - re-extract it first"

code = open("iot23_step2_detect_and_log.py").read()
code = re.sub(r"^INPUT_LOG = .*$", f'INPUT_LOG = "{matches[0]}"', code, flags=re.M)
code = re.sub(r"^ALERT_THRESHOLD = [0-9.]+", "ALERT_THRESHOLD = 0.131", code, flags=re.M)
open("iot23_step2_detect_and_log.py", "w").write(code)

%run iot23_step2_detect_and_log.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu
Loading trained model and preprocessing recipe...

Capture 'CTU-IoT-Malware-Capture-34-1' was held out (test split) - the model has not trained on it. Good.

Reading Zeek log: /content/iot23/opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-34-1/bro/conn.log.labeled
Loaded 23,145 flows.
Scoring flows...

Appending detections to: /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/detection_log.jsonl
(flows with malicious probability >= 0.131 are logged; normal traffic is not)

[2018-12-21T14:50:16+00:00] ATTACK suspected: 192.168.1.195 -> 185.244.25.235:80 (tcp/-, S0) confidence 41.1%  -> 33c775b1...
[2018-12-21T14:50:48+00:00] ATTACK suspected: 192.168.1.195 -> 185.244.25.235:80 (tcp/http, SF) confidence 99.7%  -> a3dab9b3...
[2018-12-21T14:50:50+00:00] ATTACK suspected: 192.168.1.195 -> 185.244.25.23

In [ ]:
import json, shutil
SAVE_DIR = "/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs"
LOG = f"{SAVE_DIR}/detection_log.jsonl"
BACKUP = f"{SAVE_DIR}/detection_log_backup.jsonl"

shutil.copy(LOG, BACKUP)                      # safety copy so you can restore it

lines = open(LOG).read().splitlines()
n = len(lines) // 2                           # pick an entry in the middle
entry = json.loads(lines[n])
print(f"Entry on line {n + 1} BEFORE: source IP {entry['src_ip']}, confidence {entry['confidence']}")

entry["src_ip"] = "10.0.0.99"                 # pretend someone is covering the attacker's tracks
lines[n] = json.dumps(entry)
open(LOG, "w").write("\n".join(lines) + "\n")
print(f"Entry on line {n + 1} AFTER:  source IP {entry['src_ip']}   <- edited")

Entry on line 1356 BEFORE: source IP 192.168.1.195, confidence 0.4096
Entry on line 1356 AFTER:  source IP 10.0.0.99   <- edited


In [ ]:
import re
code = open("iot23_step2_detect_and_log.py").read()
code = re.sub(r"^VERIFY_ONLY = (False|True)", "VERIFY_ONLY = True", code, flags=re.M)
open("iot23_step2_detect_and_log.py", "w").write(code)

%run iot23_step2_detect_and_log.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verifying /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/detection_log.jsonl ...
❌ Entry at line 1356 has been tampered with (its hash no longer matches).


In [1]:
"""
Smart Building Digital Forensic Readiness Framework
Step 1c (IoT-23): Rigorous cross-capture evaluation (nested leave-one-capture-out)

WHY THIS SCRIPT EXISTS
  Earlier scripts had weaknesses a reviewer would flag:
    - one random split of the captures (results swung from 0% to 99% recall)
    - the epoch / threshold were chosen on the same data that was reported
    - only one model, no baselines, one seed, no uncertainty
    - "accuracy" and pooled scores hide how badly some captures are handled

  This script fixes those:
    1. NESTED leave-one-capture-out. Each capture is the test set once. Every
       choice (early stopping epoch, alert threshold) is made ONLY from
       validation captures taken from the training side, never from the test
       capture.
    2. The alert threshold is an operating point: the score that gives a fixed
       false-alarm budget (FPR_BUDGET) on the validation captures' benign flows.
       The false-alarm rate actually obtained on the test capture is reported,
       so threshold drift between captures is visible instead of hidden.
    3. Baselines under the same protocol: logistic regression, random forest,
       gradient boosting (untuned, see notes).
    4. Several seeds (they change the inner validation captures and the model
       initialisation). Each capture is one data point; results are reported as
       the mean over captures with a bootstrap 95% confidence interval, plus
       median and worst capture.
    5. Metrics that suit imbalanced data: recall, false-alarm rate, precision,
       MCC, PR-AUC, ROC-AUC, recall at 1% / 5% false alarms (threshold-free
       view of ranking quality). Reported over DISTINCT flow patterns and also
       weighted by how many sampled flows each pattern stands for.
    6. Learning curves: training score is measured in eval mode (no dropout),
       next to the validation score, so over-fitting is visible properly.

HOW TO RUN (Colab):
    Step 1 must have been run once on the full data so the cache file exists.
    Upload this file, then:   %run iot23_step1c_rigorous_eval.py
    Quick check first: set SEEDS = [0] and MAX_FOLDS = 3 near the top.

OUTPUT FILES (in iot23_outputs/rigorous_eval/):
    results_runs.csv        one row per (model, capture, seed)
    results_per_capture.csv seed-averaged, one row per (model, capture)
    summary.csv             the table for the paper
    learning_curves.csv/png train vs validation curves of the MLP
    config.json             settings and library versions

NOTES / LIMITATIONS (state them in the paper):
  - Captures of the same malware family can sit in train and test. For a
    stricter test, hold out whole families (needs a capture -> family table
    from the IoT-23 documentation).
  - The data is a capped random sample per capture and class, so class ratios
    are not real-world ratios. Precision here would be lower in deployment.
  - Duplicate flows are collapsed (DROP_DUPLICATES). Re-run with False and
    report both; the weighted columns give a partial view of the effect.
  - Baselines use fixed default settings and were not tuned. Say so.
  - With ~10 test captures, confidence intervals are wide and significance
    tests have little power. That is a property of IoT-23, not a bug.
  - IoT-23 labels are produced per capture by the dataset authors' rules; they
    may contain label noise.

REQUIREMENTS:
    pip install torch pandas scikit-learn numpy scipy matplotlib
"""

import copy
import json
import os
import pickle
import platform
import time
import warnings

import numpy as np
import pandas as pd
import sklearn
from scipy.stats import wilcoxon
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler

try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------------------------------------------
# SETTINGS
# ----------------------------------------------------------------
OUTPUT_DIR_COLAB = "/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs"
OUTPUT_DIR_LOCAL = "./iot23_outputs"
CACHE_NAME = "iot23_sample_cap40000_chunksNone_scNone.pkl"   # made by Step 1's full run
RESULTS_SUBDIR = "rigorous_eval"

MODELS = ["mlp", "logreg", "rf", "hgb"]
SEEDS = [0, 1, 2]
MAX_FOLDS = None            # None = every eligible capture; e.g. 3 for a quick test

DROP_COLS = ["id.resp_p"]   # raw destination port differs a lot between captures; [] = keep it
DROP_DUPLICATES = True      # collapse identical flow patterns inside a capture
FPR_BUDGET = 0.05           # false-alarm budget used to set the alert threshold on validation captures
N_INNER_VAL = 3             # validation captures (taken from the training side) per fold
MIN_ROWS_EVAL = 50          # a test capture needs this many benign AND malicious patterns
MIN_ROWS_VAL = 100          # preferred minimum for validation captures

# MLP
MAX_EPOCHS = 30
PATIENCE = 5
BATCH_SIZE = 1024
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_DIMS = [128, 64]
TRAIN_AP_SAMPLE = 20000     # training rows used to measure the training score each epoch

N_BOOT = 10000              # bootstrap resamples over captures

try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    OUTPUT_DIR = OUTPUT_DIR_COLAB
else:
    OUTPUT_DIR = OUTPUT_DIR_LOCAL
RESULTS_DIR = os.path.join(OUTPUT_DIR, RESULTS_SUBDIR)
os.makedirs(RESULTS_DIR, exist_ok=True)

if "mlp" in MODELS and not HAS_TORCH:
    print("\u26a0\ufe0f  PyTorch is not installed - the MLP is skipped.")
    MODELS = [m for m in MODELS if m != "mlp"]
if HAS_TORCH:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
print(f"Models: {MODELS} | seeds: {SEEDS} | results go to: {RESULTS_DIR}")

# ----------------------------------------------------------------
# Columns and preprocessing - IDENTICAL to Step 1 / Step 2
# ----------------------------------------------------------------
NUMERIC_RAW = ["id.resp_p", "duration", "orig_bytes", "resp_bytes", "missed_bytes",
               "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes"]
LOG1P_COLS = ["duration", "orig_bytes", "resp_bytes", "missed_bytes",
              "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes"]
CAT_COLS = ["proto", "service", "conn_state"]


def numeric_frame(df):
    res = pd.DataFrame(index=df.index)
    for c in NUMERIC_RAW:
        res[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).clip(lower=0.0)
    res["history_len"] = df["history"].astype(str).str.replace("-", "", regex=False).str.len()
    for c in LOG1P_COLS:
        res[c] = np.log1p(res[c])
    return res


def make_matrix(num_part, cat_part, scaler, cat_vocab):
    parts = [scaler.transform(num_part.values)]
    names = list(num_part.columns)
    for c in CAT_COLS:
        s = cat_part[c].astype(str).values
        for v in cat_vocab[c]:
            parts.append((s == v).astype(np.float32)[:, None])
            names.append(f"{c}={v}")
    return np.hstack(parts).astype(np.float32), names


# ================================================================
# PART 1: Load the cached sample, collapse duplicates (keeping a weight)
# ================================================================
cache_path = os.path.join(OUTPUT_DIR, CACHE_NAME)
if not os.path.exists(cache_path):
    print(f"\n\u26a0\ufe0f  Cache file not found: {cache_path}")
    print("Run iot23_step1_train.py once with MAX_CHUNKS_PER_FILE = None (full data) to create it.")
    raise SystemExit("Create the cache first, then re-run.")

print(f"\nLoading cached sample from {cache_path}")
with open(cache_path, "rb") as f:
    big = pickle.load(f)["df"]
big["y"] = (big["label"].str.strip().str.lower() != "benign").astype(int)
num_df = numeric_frame(big)

weights = np.ones(len(big))
if DROP_DUPLICATES:
    before = len(big)
    key = pd.concat([num_df, big[CAT_COLS], big[["scenario", "y"]]], axis=1)
    codes, _ = pd.factorize(pd.util.hash_pandas_object(key, index=False).values)
    group_size = np.bincount(codes).astype(float)
    first = ~pd.Series(codes).duplicated().values
    weights = group_size[codes][first]          # how many sampled flows each kept pattern stands for
    big, num_df = big[first].reset_index(drop=True), num_df[first].reset_index(drop=True)
    print(f"Collapsed {before:,} sampled flows into {len(big):,} distinct patterns "
          f"(weights = number of sampled flows per pattern).")

num_v = num_df.drop(columns=DROP_COLS)
scen = big["scenario"].values
y_all = big["y"].values.astype(np.int64)
counts = (big.groupby(["scenario", "y"]).size().unstack(fill_value=0)
          .reindex(columns=[0, 1], fill_value=0))
counts.columns = ["benign", "malicious"]
print(f"Data: {len(big):,} patterns | benign {int(counts['benign'].sum()):,} | "
      f"malicious {int(counts['malicious'].sum()):,} | {len(counts)} captures")

eligible = [s for s in counts.index
            if counts.loc[s, "benign"] >= MIN_ROWS_EVAL and counts.loc[s, "malicious"] >= MIN_ROWS_EVAL]
folds = eligible[:MAX_FOLDS] if MAX_FOLDS else eligible
print(f"Test captures (each held out once): {len(folds)} -> {', '.join(folds)}")
if len(folds) < 3:
    raise SystemExit("Too few captures with both classes. Lower MIN_ROWS_EVAL.")


# ================================================================
# PART 2: Metrics
# ================================================================
def conf(y, pred, w=None):
    w = np.ones(len(y)) if w is None else w
    tp = float(w[(pred == 1) & (y == 1)].sum())
    fp = float(w[(pred == 1) & (y == 0)].sum())
    fn = float(w[(pred == 0) & (y == 1)].sum())
    tn = float(w[(pred == 0) & (y == 0)].sum())
    return tp, fp, fn, tn


def rates(tp, fp, fn, tn):
    rec = tp / (tp + fn) if tp + fn else float("nan")
    prec = tp / (tp + fp) if tp + fp else float("nan")
    fpr = fp / (fp + tn) if fp + tn else float("nan")
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) and not np.isnan(prec + rec) else 0.0
    denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = (tp * tn - fp * fn) / denom if denom > 0 else 0.0
    bal = 0.5 * (rec + (1 - fpr)) if not (np.isnan(rec) or np.isnan(fpr)) else float("nan")
    return {"recall": rec, "precision": prec, "fpr": fpr, "f1": f1, "mcc": float(mcc), "bal_acc": bal}


def recall_at_fpr(y, p, target):
    fpr, tpr, _ = roc_curve(y, p)
    return float(tpr[fpr <= target].max())


def macro_ap(y, p, groups):
    """Mean over captures of average precision (used to pick the MLP's best epoch)."""
    vals = []
    for g in np.unique(groups):
        m = groups == g
        if y[m].min() != y[m].max():
            vals.append(average_precision_score(y[m], p[m]))
    return float(np.mean(vals)) if vals else float("nan")


def threshold_from_budget(p_val, y_val, budget):
    """Alert threshold: flows scoring ABOVE it are flagged. Chosen so that at most about
    `budget` of the validation captures' benign flows are flagged. (Strictly above, so
    ties at the threshold - common when many benign flows score exactly 0 - are not flagged.)"""
    return float(np.quantile(p_val[y_val == 0], 1 - budget))


def evaluate(y, p, w, thr):
    pred = (p > thr).astype(int)
    res = rates(*conf(y, pred))
    wres = rates(*conf(y, pred, w))
    res.update({"w_recall": wres["recall"], "w_precision": wres["precision"], "w_fpr": wres["fpr"],
                "pr_auc": float(average_precision_score(y, p)),
                "pr_auc_w": float(average_precision_score(y, p, sample_weight=w)),
                "roc_auc": float(roc_auc_score(y, p)),
                "rec_at_1pct": recall_at_fpr(y, p, 0.01),
                "rec_at_5pct": recall_at_fpr(y, p, 0.05),
                "n_benign": int((y == 0).sum()), "n_malicious": int((y == 1).sum())})
    return res


# ================================================================
# PART 3: Models
# ================================================================
HISTORY = []   # MLP learning curves

if HAS_TORCH:
    class AttackDetectorMLP(nn.Module):
        def __init__(self, input_dim, hidden_dims, num_classes):
            super().__init__()
            layers, prev = [], input_dim
            for h in hidden_dims:
                layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.2)]
                prev = h
            layers.append(nn.Linear(prev, num_classes))
            self.network = nn.Sequential(*layers)

        def forward(self, x):
            return self.network(x)

    def predict_prob(model, X, batch_size=8192):
        model.eval()
        chunks = []
        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                logits = model(torch.tensor(X[i:i + batch_size]).to(device))
                chunks.append(torch.softmax(logits, dim=1)[:, 1].cpu())
        return torch.cat(chunks).numpy()

    def run_mlp(X_tr, y_tr, X_va, y_va, g_va, X_te, seed, tag):
        torch.manual_seed(seed)
        model = AttackDetectorMLP(X_tr.shape[1], HIDDEN_DIMS, 2).to(device)
        Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
        n = len(yt)
        n_pos = int((yt == 1).sum())
        n_neg = n - n_pos
        class_w = torch.tensor([n / (2 * max(n_neg, 1)), n / (2 * max(n_pos, 1))],
                               dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_w)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        sub = np.random.default_rng(seed).choice(n, size=min(TRAIN_AP_SAMPLE, n), replace=False)
        X_sub, y_sub = X_tr[sub], y_tr[sub]

        best_score, best_state, best_epoch, wait = -1.0, None, 0, 0
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            idx = torch.randperm(n)
            total = 0.0
            for i in range(0, n, BATCH_SIZE):
                b = idx[i:i + BATCH_SIZE]
                xb, yb = Xt[b].to(device), yt[b].to(device)
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()
                total += loss.item() * len(b)
            va_score = macro_ap(y_va, predict_prob(model, X_va), g_va)
            tr_score = float(average_precision_score(y_sub, predict_prob(model, X_sub)))
            HISTORY.append({**tag, "epoch": epoch, "train_loss": total / n,
                            "train_ap_eval_mode": tr_score, "val_macro_ap": va_score})
            if best_state is None or va_score > best_score + 1e-4:
                best_score, best_epoch, wait = va_score, epoch, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                wait += 1
                if wait >= PATIENCE:
                    break
        model.load_state_dict(best_state)
        return predict_prob(model, X_va), predict_prob(model, X_te), {"best_epoch": best_epoch}


def run_sklearn(name, X_tr, y_tr, X_va, X_te, seed):
    n = len(y_tr)
    n_pos = int((y_tr == 1).sum())
    n_neg = n - n_pos
    if name == "logreg":
        clf = LogisticRegression(max_iter=2000, class_weight="balanced")
        clf.fit(X_tr, y_tr)
    elif name == "rf":
        clf = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, max_samples=0.5, n_jobs=-1,
                                     class_weight="balanced_subsample", random_state=seed)
        clf.fit(X_tr, y_tr)
    elif name == "hgb":
        sw = np.where(y_tr == 1, n / (2 * max(n_pos, 1)), n / (2 * max(n_neg, 1)))
        clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=seed)
        clf.fit(X_tr, y_tr, sample_weight=sw)
    else:
        raise ValueError(name)
    return clf.predict_proba(X_va)[:, 1], clf.predict_proba(X_te)[:, 1], {"best_epoch": np.nan}


# ================================================================
# PART 4: Nested leave-one-capture-out
# ================================================================
records = []
t_start = time.time()
print("\nRunning nested leave-one-capture-out (every choice is made without the test capture)...\n")

for fi, g in enumerate(folds):
    te_rows = scen == g
    for seed in SEEDS:
        rng = np.random.default_rng([seed, fi])
        pool = [s for s in eligible if s != g]
        good = [s for s in pool if counts.loc[s, "benign"] >= MIN_ROWS_VAL
                and counts.loc[s, "malicious"] >= MIN_ROWS_VAL]
        cands = good if len(good) >= N_INNER_VAL else pool
        val_caps = list(rng.choice(cands, size=min(N_INNER_VAL, len(cands)), replace=False))
        va_rows = np.isin(scen, val_caps)
        tr_rows = ~te_rows & ~va_rows

        # preprocessing is fitted on the inner-training captures only
        scaler = StandardScaler().fit(num_v[tr_rows].values)
        cat_vocab = {c: big.loc[tr_rows, c].astype(str).value_counts().head(30).index.tolist()
                     for c in CAT_COLS}
        X_all, _ = make_matrix(num_v, big, scaler, cat_vocab)
        X_tr, y_tr = X_all[tr_rows], y_all[tr_rows]
        X_va, y_va, g_va = X_all[va_rows], y_all[va_rows], scen[va_rows]
        X_te, y_te, w_te = X_all[te_rows], y_all[te_rows], weights[te_rows]

        for name in MODELS:
            t0 = time.time()
            if name == "mlp":
                p_va, p_te, extra = run_mlp(X_tr, y_tr, X_va, y_va, g_va, X_te, seed,
                                            {"capture": g, "seed": seed})
            else:
                p_va, p_te, extra = run_sklearn(name, X_tr, y_tr, X_va, X_te, seed)
            thr = threshold_from_budget(p_va, y_va, FPR_BUDGET)
            rec = evaluate(y_te, p_te, w_te, thr)
            rec.update({"model": name, "capture": g, "seed": seed, "threshold": thr,
                        "val_captures": "|".join(map(str, val_caps)), **extra})
            records.append(rec)
            print(f"[{fi + 1}/{len(folds)}] {g} seed {seed} {name:6s}: recall {rec['recall']:.3f} | "
                  f"false-alarm rate {rec['fpr']:.3f} | PR-AUC {rec['pr_auc']:.3f} ({time.time() - t0:.0f}s)")

print(f"\nFinished in {(time.time() - t_start) / 60:.1f} min.")

# ================================================================
# PART 5: Aggregate - each capture is one data point
# ================================================================
runs = pd.DataFrame(records)
METRICS = ["recall", "fpr", "precision", "f1", "mcc", "bal_acc", "pr_auc", "roc_auc",
           "rec_at_1pct", "rec_at_5pct", "w_recall", "w_fpr", "w_precision", "pr_auc_w"]
per_cap = runs.groupby(["model", "capture"])[METRICS + ["n_benign", "n_malicious", "threshold"]] \
    .mean().reset_index()


def boot_ci(values, n_boot=N_BOOT, seed=0):
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    if len(v) < 2:
        return float("nan"), float("nan")
    idx = np.random.default_rng(seed).integers(0, len(v), (n_boot, len(v)))
    lo, hi = np.percentile(v[idx].mean(axis=1), [2.5, 97.5])
    return float(lo), float(hi)


rows = []
for name in MODELS:
    sub = per_cap[per_cap["model"] == name]
    for m in METRICS:
        v = sub[m].values
        lo, hi = boot_ci(v)
        seed_means = runs[runs["model"] == name].groupby("seed")[m].mean()
        rows.append({"model": name, "metric": m, "mean_over_captures": float(np.nanmean(v)),
                     "ci95_low": lo, "ci95_high": hi, "median": float(np.nanmedian(v)),
                     "worst_capture": float(np.nanmin(v)), "best_capture": float(np.nanmax(v)),
                     "std_across_seeds": float(seed_means.std(ddof=0)) if len(seed_means) > 1 else float("nan")})
summary = pd.DataFrame(rows)

KEY = ["recall", "fpr", "precision", "mcc", "pr_auc", "rec_at_5pct", "w_recall"]
print("\n" + "=" * 78)
print(f"RESULTS: mean over {len(folds)} held-out captures [95% bootstrap CI over captures]")
print(f"Alert threshold = score giving {FPR_BUDGET:.0%} false alarms on validation captures. "
      f"'fpr' is what the test capture really got.")
print("=" * 78)
for name in MODELS:
    print(f"\n{name}")
    for m in KEY:
        r = summary[(summary.model == name) & (summary.metric == m)].iloc[0]
        print(f"  {m:12s} {r.mean_over_captures:6.3f}  [{r.ci95_low:.3f}, {r.ci95_high:.3f}]   "
              f"median {r['median']:.3f}   worst {r.worst_capture:.3f}   seed-std {r.std_across_seeds:.3f}")

print("\nPer-capture results (mean over seeds):")
for name in MODELS:
    print(f"\n{name}")
    cols = ["capture", "n_benign", "n_malicious", "recall", "fpr", "precision", "pr_auc", "rec_at_5pct", "w_recall"]
    print(per_cap[per_cap.model == name][cols].round(3).to_string(index=False))

# paired comparison against the first model (captures are the pairs)
if len(MODELS) > 1:
    print("\nPaired Wilcoxon signed-rank tests over captures (first model vs each other model):")
    wide = {m: per_cap.pivot(index="capture", columns="model", values=m) for m in ("recall", "pr_auc")}
    for other in MODELS[1:]:
        for m in ("recall", "pr_auc"):
            d = wide[m][MODELS[0]] - wide[m][other]
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    p = wilcoxon(d.values).pvalue
                print(f"  {MODELS[0]} vs {other:7s} {m:7s}: mean diff {d.mean():+.3f}, p = {p:.3f}")
            except ValueError:
                print(f"  {MODELS[0]} vs {other:7s} {m:7s}: mean diff {d.mean():+.3f}, test not possible (identical results)")
    print(f"  (only {len(folds)} captures: low power, do not over-interpret p-values)")

# ================================================================
# PART 6: Learning curves (training score in eval mode vs validation)
# ================================================================
if HISTORY:
    hist = pd.DataFrame(HISTORY)
    hist.to_csv(os.path.join(RESULTS_DIR, "learning_curves.csv"), index=False)
    curve = hist.groupby("epoch").agg(runs=("capture", "size"), train_ap=("train_ap_eval_mode", "mean"),
                                      val_ap=("val_macro_ap", "mean"), loss=("train_loss", "mean"))
    print("\nMLP learning curves (mean over runs still training at that epoch):")
    print(curve.round(3).to_string())
    print(f"Best epochs chosen by early stopping: median {int(runs[runs.model == 'mlp']['best_epoch'].median())}, "
          f"range {int(runs[runs.model == 'mlp']['best_epoch'].min())}-{int(runs[runs.model == 'mlp']['best_epoch'].max())}")
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        keep = curve[curve["runs"] >= 0.5 * curve["runs"].max()]
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(keep.index, keep["train_ap"], label="training (eval mode)")
        ax.plot(keep.index, keep["val_ap"], label="validation captures")
        ax.set_xlabel("epoch")
        ax.set_ylabel("average precision")
        ax.legend()
        fig.tight_layout()
        fig.savefig(os.path.join(RESULTS_DIR, "learning_curves.png"), dpi=200)
        print("Saved learning_curves.png")
    except ImportError:
        print("matplotlib not installed - curve plot skipped (the CSV was saved).")

# ================================================================
# PART 7: Save everything
# ================================================================
runs.to_csv(os.path.join(RESULTS_DIR, "results_runs.csv"), index=False)
per_cap.to_csv(os.path.join(RESULTS_DIR, "results_per_capture.csv"), index=False)
summary.to_csv(os.path.join(RESULTS_DIR, "summary.csv"), index=False)
config = {"models": MODELS, "seeds": SEEDS, "drop_cols": DROP_COLS, "drop_duplicates": DROP_DUPLICATES,
          "fpr_budget": FPR_BUDGET, "n_inner_val": N_INNER_VAL, "min_rows_eval": MIN_ROWS_EVAL,
          "min_rows_val": MIN_ROWS_VAL, "max_epochs": MAX_EPOCHS, "patience": PATIENCE,
          "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
          "hidden_dims": HIDDEN_DIMS, "test_captures": folds, "cache": CACHE_NAME,
          "versions": {"python": platform.python_version(), "numpy": np.__version__,
                       "pandas": pd.__version__, "sklearn": sklearn.__version__,
                       "torch": torch.__version__ if HAS_TORCH else None}}
with open(os.path.join(RESULTS_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print(f"\n\u2705 Saved to {RESULTS_DIR}: results_runs.csv, results_per_capture.csv, summary.csv, config.json"
      + (", learning_curves.csv/png" if HISTORY else ""))
print("\nHow to report: use summary.csv (mean over captures with CI), show the per-capture table, state the")
print("false-alarm budget, and say plainly which captures fail. Do not report pooled or best-split numbers.")

Mounted at /content/drive
Using device: cpu
Models: ['mlp', 'logreg', 'rf', 'hgb'] | seeds: [0, 1, 2] | results go to: /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/rigorous_eval

Loading cached sample from /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/iot23_sample_cap40000_chunksNone_scNone.pkl
Collapsed 886,853 sampled flows into 153,180 distinct patterns (weights = number of sampled flows per pattern).
Data: 153,180 patterns | benign 84,981 | malicious 68,199 | 23 captures
Test captures (each held out once): 10 -> CTU-IoT-Malware-Capture-1-1, CTU-IoT-Malware-Capture-17-1, CTU-IoT-Malware-Capture-3-1, CTU-IoT-Malware-Capture-34-1, CTU-IoT-Malware-Capture-35-1, CTU-IoT-Malware-Capture-48-1, CTU-IoT-Malware-Capture-49-1, CTU-IoT-Malware-Capture-52-1, CTU-IoT-Malware-Capture-7-1, CTU-IoT-Malware-Capture-8-1

Running nested leave-one-capture-out (every choice is made without the test capture)...

[1/10] CTU-IoT-Malware-Capture-1-1 s